In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import pandas as pd
from sklearn.metrics import classification_report

from src.modeling import evaluate_model, get_models, build_pipeline
from src.preprocessing import add_features, clean_data, load_data, split_data

DATA_PATH = Path("../data/raw/Churn_Modelling.csv")

In [2]:
df = load_data(DATA_PATH)
df = clean_data(df)
df = add_features(df)

X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

In [3]:
results = []

for model_name, model in get_models().items():
    pipeline = build_pipeline(model, X_train)
    pipeline.fit(X_train, y_train)
    metrics = evaluate_model(pipeline, X_val, y_val)

    row = {"model": model_name}
    row.update(metrics)
    results.append(row)

results_df = pd.DataFrame(results).sort_values(by="f1", ascending=False).reset_index(drop=True)
results_df

,model,accuracy,precision,recall,f1,roc_auc
0,xgboost,0.838108,0.589744,0.676471,0.630137,0.864260
1,random_forest,0.830779,0.571823,0.676471,0.619760,0.860691
2,gradient_boosting,0.864091,0.752475,0.496732,0.598425,0.865798
3,decision_tree,0.782811,0.478448,0.725490,0.576623,0.838810
4,knn,0.839440,0.681564,0.398693,0.503093,0.801633
5,logistic_regression,0.730180,0.401590,0.660131,0.499382,0.770566


In [4]:
results_df.to_csv("../models/experiment_results.csv", index=False)
results_df

,model,accuracy,precision,recall,f1,roc_auc
0,xgboost,0.838108,0.589744,0.676471,0.630137,0.864260
1,random_forest,0.830779,0.571823,0.676471,0.619760,0.860691
2,gradient_boosting,0.864091,0.752475,0.496732,0.598425,0.865798
3,decision_tree,0.782811,0.478448,0.725490,0.576623,0.838810
4,knn,0.839440,0.681564,0.398693,0.503093,0.801633
5,logistic_regression,0.730180,0.401590,0.660131,0.499382,0.770566


In [5]:
best_model_name = results_df.iloc[0]["model"]
best_model_name

'xgboost'

In [6]:
models = get_models()
best_model = build_pipeline(models[best_model_name], X_train)
best_model.fit(X_train, y_train)

test_metrics = evaluate_model(best_model, X_test, y_test)
test_metrics

{'accuracy': 0.8226666666666667,
 'precision': 0.5543478260869565,
 'recall': 0.6666666666666666,
 'f1': 0.6053412462908012,
 'roc_auc': 0.8574599577407737}

In [7]:
y_test_pred = best_model.predict(X_test)
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.91      0.86      0.89      1194
           1       0.55      0.67      0.61       306

    accuracy                           0.82      1500
   macro avg       0.73      0.76      0.75      1500
weighted avg       0.84      0.82      0.83      1500



In [8]:
import joblib

joblib.dump(best_model, "../models/best_model.pkl")

['../models/best_model.pkl']